레이블
1이 상장폐지
0이 상장폐지 아님

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

df = pd.read_excel("train.xlsx")
df = df.to_csv("train.csv")
df = pd.read_csv('train.csv')
df = df.drop(columns = 'Unnamed: 0', errors = 'ignore')
print(df)

        종목코드           총자산    총자본회전율       부채비율    매출액증가율      자기자본회전율  \
0        360   6130.333968  0.846342 -32.596321 -0.036030 -2674.128824   
1        420    719.682766  0.541509  -5.124169 -0.504883  -223.327431   
2        790   2062.599327  1.009479   4.145668  0.311749   519.444543   
3        800  18007.307005  0.978722   2.499255  0.751287   342.479670   
4        895   2098.958240  0.583371   1.620366  0.022887   152.864625   
...      ...           ...       ...        ...       ...          ...   
1795  227840           NaN       NaN        NaN       NaN          NaN   
1796  229640           NaN       NaN        NaN       NaN          NaN   
1797  234080           NaN       NaN        NaN       NaN          NaN   
1798  241560           NaN       NaN        NaN       NaN          NaN   
1799  900100           NaN       NaN        NaN       NaN          NaN   

           당기순이익  최대주주변경  대표이사변경  전환사채  파산신청  거래정지  불성실공시법인  레이블  
0    -661.336250       0       0     0     0

In [15]:
import pandas as pd

df = pd.read_excel("test_예시.xlsx")
df = df.to_csv("text_예시.csv")
df = pd.read_csv("text_예시.csv")
df = df.drop(columns = 'Unnamed: 0', errors = 'ignore')

print(df)

    종목코드          총자산    총자본회전율       부채비율    매출액증가율      자기자본회전율       당기순이익  \
0   4230   428.244201  0.145047   2.504589 -0.474146    50.832895 -110.196437   
1   8400  4235.038475  0.230752  87.004072 -0.219220  2030.707545 -609.697673   
2  11720   401.758378  0.726854   2.599183  0.221462   261.607927  -53.147792   

   최대주주변경  대표이사변경  전환사채  파산신청  거래정지  불성실공시법인  레이블  
0       4       0    12     0     4        9    1  
1       0       0     0     2     3        4    1  
2       6       0     7     0     1        4    1  


In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

# 1. train.csv 데이터 로드
df1 = pd.read_csv('train.csv')
df1 = df1.dropna()

# 2. 데이터(X)와 타겟(y) 분리 및 설정 (기본값은 마지막 열이 타겟)
target_col_name = '레이블'  # <-- 시험지에 적힌 정확한 타겟 컬럼명 입력
X_train = df1.drop(columns=[target_col_name, '종목코드','Unnamed: 0' ], errors='ignore')
y_train = df1[target_col_name]



In [114]:
# 4. 종합 연관성 순위 계산 및 상위 5개 '위치(인덱스 번호)' 추출
corr_rank = X_train.corrwith(y_train).abs().rank(ascending=False).sort_values(ascending=True)

rf_final = RandomForestClassifier(n_estimators=50, class_weight='balanced', n_jobs=-1, random_state=40)
rf_final.fit(X_train.values, y_train.values) # .values로 이름 경고 방지
rf_rank = pd.Series(rf_final.feature_importances_, index=X_train.columns).rank(ascending=False).sort_values(ascending=True)

In [115]:
# 종합 점수가 가장 낮은(우수한) 5개 컬럼의 실제 위치 번호(정수 인덱스)를 리스트로 추출
top_5_indices = (corr_rank + (rf_rank)).sort_values(ascending=False).index[:5]
top_5_pos = [X_train.columns.get_loc(col) for col in top_5_indices]

# 5. 모델 학습 진행 (.values로 컬럼명 완전 제거)
X_train_final = X_train[top_5_indices].values 

final_scaler = StandardScaler()
X_train_scaled = final_scaler.fit_transform(X_train_final)

In [117]:
best_model = RandomForestClassifier(
    n_estimators=300,            # 트리 개수를 확장하여 일반화 성능 극대화
    criterion='entropy',         # 불균형 데이터 대응력 강화
    max_features='sqrt',         # 정예 5개 변수 시너지 최적화
    min_samples_split=5,         # 분할 최소 샘플 수 제한으로 과적합 방지
    min_samples_leaf=2,          # 리프 노드 제한으로 처음 보는 데이터 방어력 증대
    class_weight='balanced',     # 소수 클래스 가중치 부여로 Macro F1 점수 사수
    n_jobs=-1, 
    random_state=40
)
best_model.fit(X_train_scaled, y_train.values) # 타겟에서도 이름을 제거하여 일관성 유지

from sklearn.metrics import classification_report

pred = best_model.predict(X_train_scaled)
print(classification_report(y_train, pred))

# 6. 규칙 파일 보관
joblib.dump(best_model, 'best_rf_model.pkl')
joblib.dump(final_scaler, 'final_scaler.pkl')
joblib.dump(top_5_indices, 'top_5_indices.pkl')

print(f"▶ 선정된 열 위치 번호: {top_5_pos}")

              precision    recall  f1-score   support

           0       0.99      0.98      0.98       674
           1       0.94      0.96      0.95       226

    accuracy                           0.97       900
   macro avg       0.96      0.97      0.96       900
weighted avg       0.97      0.97      0.97       900

▶ 선정된 열 위치 번호: [4, 9, 8, 3, 2]


In [128]:
from sklearn.metrics import classification_report
df_test = pd.read_excel('test.xlsx')
df_test = df_test.to_csv("test.csv")
df_test = pd.read_csv("test.csv")
df_test = df_test.drop(columns = 'Unnamed: 0', errors = 'ignore')

df_test_features = df_test.copy()
df_test_features = df_test_features.drop(columns=[target_col_name, '종목코드','Unnamed: 0' ], errors='ignore')
df_test_target = df_test.레이블
   
loaded_idices = joblib.load('top_5_indices.pkl')
X_realtime = df_test_features[top_5_indices].values 
    
loaded_scaler = joblib.load('final_scaler.pkl')
loaded_model = joblib.load('best_rf_model.pkl')

X_realtime_scaled = loaded_scaler.fit_transform(X_realtime)
    
pred = loaded_model.predict(X_realtime_scaled)

print(classification_report(df_test_target, pred))

              precision    recall  f1-score   support

           0       0.92      0.71      0.80        76
           1       0.46      0.79      0.58        24

    accuracy                           0.73       100
   macro avg       0.69      0.75      0.69       100
weighted avg       0.81      0.73      0.75       100



In [134]:
best_model = RandomForestClassifier(random_state=40)
best_model.fit(X_train, y_train.values) # 타겟에서도 이름을 제거하여 일관성 유지

from sklearn.metrics import classification_report

pred = best_model.predict(X_train)
print(classification_report(y_train, pred))

# 6. 규칙 파일 보관
joblib.dump(best_model, 'best_rf_model.pkl')
joblib.dump(final_scaler, 'final_scaler.pkl')
joblib.dump(top_5_indices, 'top_5_indices.pkl')

print(f"▶ 선정된 열 위치 번호: {top_5_pos}")

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       674
           1       1.00      1.00      1.00       226

    accuracy                           1.00       900
   macro avg       1.00      1.00      1.00       900
weighted avg       1.00      1.00      1.00       900

▶ 선정된 열 위치 번호: [4, 9, 8, 3, 2]


In [ ]:
from sklearn.metrics import classification_report
df_test = pd.read_excel('test.xlsx')
df_test = df_test.to_csv("test.csv")
df_test = pd.read_csv("test.csv")
df_test = df_test.drop(columns = 'Unnamed: 0', errors = 'ignore')

df_test_features = df_test.copy()
df_test_features = df_test_features.drop(columns=[target_col_name, '종목코드','Unnamed: 0' ], errors='ignore')
df_test_target = df_test.레이블
   
loaded_idices = joblib.load('top_5_indices.pkl')
X_realtime = df_test_features[loaded_idices].values 
    
loaded_scaler = joblib.load('final_scaler.pkl')
loaded_model = joblib.load('best_rf_model.pkl')

X_realtime_scaled = loaded_scaler.fit_transform(X_realtime)

pred = loaded_model.predict(df_test_features)

print(classification_report(df_test_target, pred))

              precision    recall  f1-score   support

           0       0.99      0.97      0.98        76
           1       0.92      0.96      0.94        24

    accuracy                           0.97       100
   macro avg       0.95      0.97      0.96       100
weighted avg       0.97      0.97      0.97       100

